In [0]:

CREATE SCHEMA IF NOT EXISTS silver;


/* ---------------------------------------------------------
   1. Parse, standardize, validate, and rank the Bronze data
   --------------------------------------------------------- */

CREATE OR REPLACE TEMP VIEW crypto_candles_validated AS

WITH typed_data AS (
    SELECT
        /* Preserve raw values for rejected-record analysis */
        `close` AS raw_close,
        `high` AS raw_high,
        `low` AS raw_low,
        `open` AS raw_open,
        product_id AS raw_product_id,
        `timestamp` AS raw_timestamp,
        volume AS raw_volume,
        trade_timestamp AS raw_trade_timestamp,
        trade_date AS raw_trade_date,
        _ingested_at AS raw_ingested_at,
        _source AS raw_source,

        /* Standardized identifiers */
        UPPER(TRIM(product_id)) AS product_id,

        SPLIT(UPPER(TRIM(product_id)), '-')[0]
            AS base_asset_symbol,

        SPLIT(UPPER(TRIM(product_id)), '-')[1]
            AS quote_asset_symbol,

        UPPER(TRIM(_source)) AS source_system,

        /* Typed timestamps */
        TRY_CAST(`timestamp` AS BIGINT)
            AS raw_epoch_seconds,

        TRY_CAST(trade_timestamp AS TIMESTAMP)
            AS parsed_trade_timestamp,

        TRY_CAST(trade_date AS DATE)
            AS parsed_trade_date,

        TRY_CAST(_ingested_at AS TIMESTAMP)
            AS source_ingested_at,

        /* Typed market values */
        TRY_CAST(`open` AS DECIMAL(38, 10))
            AS open_price,

        TRY_CAST(`high` AS DECIMAL(38, 10))
            AS high_price,

        TRY_CAST(`low` AS DECIMAL(38, 10))
            AS low_price,

        TRY_CAST(`close` AS DECIMAL(38, 10))
            AS close_price,

        TRY_CAST(volume AS DECIMAL(38, 10))
            AS volume

    FROM bronze.coinbase_crypto_candles_raw
),

ranked_data AS (
    SELECT
        *,

        ROW_NUMBER() OVER (
            PARTITION BY
                product_id,
                parsed_trade_timestamp,
                source_system
            ORDER BY source_ingested_at DESC
        ) AS duplicate_rank

    FROM typed_data
),

classified_data AS (
    SELECT
        *,

        CASE
            WHEN product_id IS NULL OR product_id = ''
                THEN 'MISSING_PRODUCT_ID'

            WHEN product_id NOT RLIKE '^[A-Z0-9]+-[A-Z0-9]+$'
                THEN 'INVALID_PRODUCT_ID'

            WHEN source_system IS NULL OR source_system = ''
                THEN 'MISSING_SOURCE_SYSTEM'

            WHEN raw_epoch_seconds IS NULL
                THEN 'INVALID_EPOCH_TIMESTAMP'

            WHEN parsed_trade_timestamp IS NULL
                THEN 'INVALID_TRADE_TIMESTAMP'

            WHEN parsed_trade_date IS NULL
                THEN 'INVALID_TRADE_DATE'

            WHEN source_ingested_at IS NULL
                THEN 'INVALID_INGESTION_TIMESTAMP'

            WHEN open_price IS NULL
                THEN 'INVALID_OPEN_PRICE'

            WHEN high_price IS NULL
                THEN 'INVALID_HIGH_PRICE'

            WHEN low_price IS NULL
                THEN 'INVALID_LOW_PRICE'

            WHEN close_price IS NULL
                THEN 'INVALID_CLOSE_PRICE'

            WHEN volume IS NULL
                THEN 'INVALID_VOLUME'

            WHEN open_price <= 0
              OR high_price <= 0
              OR low_price <= 0
              OR close_price <= 0
                THEN 'NONPOSITIVE_PRICE'

            WHEN volume < 0
                THEN 'NEGATIVE_VOLUME'

            WHEN high_price < GREATEST(
                open_price,
                close_price,
                low_price
            )
                THEN 'HIGH_PRICE_BELOW_OTHER_PRICES'

            WHEN low_price > LEAST(
                open_price,
                close_price,
                high_price
            )
                THEN 'LOW_PRICE_ABOVE_OTHER_PRICES'

            WHEN parsed_trade_date <> TO_DATE(parsed_trade_timestamp)
                THEN 'TRADE_DATE_TIMESTAMP_MISMATCH'

            WHEN ABS(
                UNIX_TIMESTAMP(parsed_trade_timestamp)
                - raw_epoch_seconds
            ) > 1
                THEN 'EPOCH_TIMESTAMP_MISMATCH'

            WHEN duplicate_rank > 1
                THEN 'DUPLICATE_CANDLE'

            ELSE NULL
        END AS rejection_reason

    FROM ranked_data
)

SELECT *
FROM classified_data;


/* ------------------------------------------
   2. Create the clean Silver candle table
   ------------------------------------------ */

CREATE OR REPLACE TABLE silver.crypto_candles_clean
USING DELTA
AS

SELECT
    SHA2(
        CONCAT_WS(
            '|',
            product_id,
            CAST(parsed_trade_timestamp AS STRING),
            source_system
        ),
        256
    ) AS candle_key,

    product_id,
    base_asset_symbol,
    quote_asset_symbol,

    parsed_trade_timestamp AS trade_timestamp,
    parsed_trade_date AS trade_date,

    open_price,
    high_price,
    low_price,
    close_price,
    volume,

    source_system,
    source_ingested_at,

    CURRENT_TIMESTAMP() AS silver_processed_at

FROM crypto_candles_validated

WHERE rejection_reason IS NULL;


/* ---------------------------------------------
   3. Create the rejected Silver candle table
   --------------------------------------------- */

CREATE OR REPLACE TABLE silver.crypto_candles_rejected
USING DELTA
AS

SELECT
    /* Original Bronze values */
    raw_close,
    raw_high,
    raw_low,
    raw_open,
    raw_product_id,
    raw_timestamp,
    raw_volume,
    raw_trade_timestamp,
    raw_trade_date,
    raw_ingested_at,
    raw_source,

    /* Parsed values for troubleshooting */
    product_id,
    base_asset_symbol,
    quote_asset_symbol,
    raw_epoch_seconds,
    parsed_trade_timestamp,
    parsed_trade_date,
    open_price,
    high_price,
    low_price,
    close_price,
    volume,
    source_system,
    source_ingested_at,

    duplicate_rank,
    rejection_reason,

    CURRENT_TIMESTAMP() AS silver_rejected_at

FROM crypto_candles_validated

WHERE rejection_reason IS NOT NULL;